In [1]:
import warnings
import sys
import csv
import os
import pandas as pd

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px

pd.options.display.float_format = '{:,.3f}'.format

import plotly.graph_objects as go
import string
from nltk.corpus import stopwords
import re

In [2]:
df = pd.read_excel(r"D:\PROJECT ML - MPP SHIFT\dataset\dataset.xlsx")

In [3]:
df

,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,JUMLAH_STRUK_SHIFT_SIANG,PERCENT_ONTIME_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_SIANG,AVG_SLA_SHIFT_PAGI,AVG_SLA_SHIFT_SIANG,QTY_PER_STRUK_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_SIANG,SALES_SHIFT_PAGI,SALES_SHIFT_SIANG,KURIR_PAGI,KURIR_SIANG
0,1,10,1M6U,142,148,0.204,0.365,124.204,79.306,7.563,6.612,"15,144,114.560","13,628,765.740",3,6
1,1,10,1M6V,166,160,0.163,0.287,138.476,107.974,10.988,9.338,"21,941,389.670","16,598,755.790",3,4
2,1,10,JD75,187,132,0.289,0.462,90.984,65.287,8.027,7.070,"23,167,740.650","12,373,117.150",5,4
3,1,10,JD76,92,68,1.000,0.971,29.174,31.448,10.707,11.567,"13,081,894.560","8,977,878.370",4,4
4,1,10,KF18,143,161,0.650,0.354,53.154,75.117,11.245,7.182,"16,707,311.540","11,379,151.400",2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,31,10,TI12,143,132,0.119,0.114,195.476,183.078,8.601,8.884,"15,365,619.890","12,882,834.210",5,6
808,31,10,TI17,233,179,0.206,0.341,144.871,94.667,8.940,7.276,"21,441,707.310","13,455,720.620",6,5
809,31,10,UD92,159,125,0.151,0.184,139.767,111.154,8.793,9.094,"16,802,278.110","11,546,145.030",4,4
810,31,10,UE21,107,177,1.000,0.994,26.907,22.722,8.906,6.210,"9,476,654.900","12,102,683.840",5,4


In [4]:
df[df['KODE_TOKO'] == '1M6U'].sort_values(by=['BULAN', 'TANGGAL']).head(15)

,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,JUMLAH_STRUK_SHIFT_SIANG,PERCENT_ONTIME_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_SIANG,AVG_SLA_SHIFT_PAGI,AVG_SLA_SHIFT_SIANG,QTY_PER_STRUK_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_SIANG,SALES_SHIFT_PAGI,SALES_SHIFT_SIANG,KURIR_PAGI,KURIR_SIANG
0,1,10,1M6U,142,148,0.204,0.365,124.204,79.306,7.563,6.612,"15,144,114.560","13,628,765.740",3,6
28,2,10,1M6U,125,49,0.336,0.490,96.800,64.130,8.712,7.413,"14,063,093.760","4,564,065.120",5,5
56,3,10,1M6U,146,114,0.363,0.465,87.952,76.690,7.267,5.965,"14,825,449.600","10,339,877.350",4,3
84,4,10,1M6U,147,92,0.360,0.467,89.156,71.656,9.007,6.056,"20,113,450.420","7,530,965.810",3,5
112,5,10,1M6U,150,118,0.253,0.381,118.487,83.518,8.240,7.746,"14,363,115.510","11,384,073.020",5,4
140,6,10,1M6U,143,125,0.161,0.320,141.133,94.615,10.413,8.246,"17,246,404.210","11,208,222.440",4,3
168,7,10,1M6U,114,116,0.333,0.578,102.263,56.167,8.553,5.702,"14,017,701.050","10,778,114.470",2,6
196,8,10,1M6U,112,103,0.482,1.000,70.000,36.932,6.295,5.990,"10,882,341.450","9,681,593.740",4,5
224,9,10,1M6U,116,127,0.362,0.409,77.612,69.724,6.078,7.236,"10,794,298.370","12,883,703.780",4,4
252,10,10,1M6U,136,119,0.338,0.471,100.765,70.076,5.831,5.398,"11,063,636.160","10,240,687.430",3,4


In [5]:
# Ambil nilai unik di kolom 'Category' dan urutkan
unique_categories = df['KODE_TOKO'].unique()  # Ambil nilai unik
sorted_categories = sorted(unique_categories)  # Urutkan nilai unik

In [6]:
# Buat mapping berdasarkan urutan yang sudah di-sort
category_mapping = {category: index + 1 for index, category in enumerate(sorted_categories)}
print(category_mapping)

{'1M6U': 1, '1M6V': 2, 'JD75': 3, 'JD76': 4, 'KF18': 5, 'KF65': 6, 'PA39': 7, 'TH06': 8, 'TH72': 9, 'TI12': 10, 'TI17': 11, 'UD92': 12, 'UE21': 13, 'W785': 14}


In [7]:
df['KODE_TOKO'] = df['KODE_TOKO'].map(category_mapping)
df

,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,JUMLAH_STRUK_SHIFT_SIANG,PERCENT_ONTIME_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_SIANG,AVG_SLA_SHIFT_PAGI,AVG_SLA_SHIFT_SIANG,QTY_PER_STRUK_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_SIANG,SALES_SHIFT_PAGI,SALES_SHIFT_SIANG,KURIR_PAGI,KURIR_SIANG
0,1,10,1,142,148,0.204,0.365,124.204,79.306,7.563,6.612,"15,144,114.560","13,628,765.740",3,6
1,1,10,2,166,160,0.163,0.287,138.476,107.974,10.988,9.338,"21,941,389.670","16,598,755.790",3,4
2,1,10,3,187,132,0.289,0.462,90.984,65.287,8.027,7.070,"23,167,740.650","12,373,117.150",5,4
3,1,10,4,92,68,1.000,0.971,29.174,31.448,10.707,11.567,"13,081,894.560","8,977,878.370",4,4
4,1,10,5,143,161,0.650,0.354,53.154,75.117,11.245,7.182,"16,707,311.540","11,379,151.400",2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,31,10,10,143,132,0.119,0.114,195.476,183.078,8.601,8.884,"15,365,619.890","12,882,834.210",5,6
808,31,10,11,233,179,0.206,0.341,144.871,94.667,8.940,7.276,"21,441,707.310","13,455,720.620",6,5
809,31,10,12,159,125,0.151,0.184,139.767,111.154,8.793,9.094,"16,802,278.110","11,546,145.030",4,4
810,31,10,13,107,177,1.000,0.994,26.907,22.722,8.906,6.210,"9,476,654.900","12,102,683.840",5,4


In [8]:
df['jumlah_kurir'] = df['KURIR_PAGI'] + df['KURIR_SIANG']

df = df[[
    'TANGGAL', 'BULAN', 'KODE_TOKO', 'JUMLAH_STRUK_SHIFT_PAGI',
       'PERCENT_ONTIME_SHIFT_PAGI', 'AVG_SLA_SHIFT_PAGI',
       'QTY_PER_STRUK_SHIFT_PAGI', 'SALES_SHIFT_PAGI',
       'KURIR_PAGI','jumlah_kurir'
]]

In [9]:
X = df.drop(columns='KURIR_PAGI')
y = df[['KURIR_PAGI']]
y

,KURIR_PAGI
0,3
1,3
2,5
3,4
4,2
...,...
807,5
808,6
809,4
810,5


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
len(X_train), len(X_test)

(544, 268)

In [11]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement xgboost (from versions: none)
ERROR: No matching distribution found for xgboost


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

reg = XGBRegressor(random_state=42).fit(X_train, y_train)
reg.score(X, y)

ModuleNotFoundError: No module named 'xgboost'